In [1]:
pip install numpy pandas scikit-learn scipy matplotlib seaborn GEOparse

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
Distributed Consensus Clustering using K-means, Spectral Clustering, and ADMM
Fixed version — corrects:
  1. Hardcoded Windows paths with invalid escape sequences
  2. ADMM x-update bug causing instant (fake) convergence at iteration 1
  3. O(n^2) co-association matrix loop -> vectorised numpy
  4. true_labels alignment with interleaved batch indices
  5. Silhouette score uses 1-Z (distance) from consensus matrix instead of raw features
"""

import numpy as np
import pandas as pd
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score, silhouette_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

# ─────────────────────────────────────────────
# CONFIG  — use relative paths, no hardcoded drive letters
# ─────────────────────────────────────────────
DATA_DIR   = "data"
PLOTS_DIR  = os.path.join("data", "plots")
N_CLUSTERS = 6
N_ITER     = 50
RHO        = 1.0

os.makedirs(PLOTS_DIR, exist_ok=True)


# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
def load_batches():
    batches      = {}
    batch_indices = {}   # FIX: track which original sample indices belong to each batch

    labels_df       = pd.read_csv(os.path.join(DATA_DIR, "true_labels.csv"), index_col=0)
    all_sample_ids  = labels_df.index.tolist()          # original order from step 1

    for fname in sorted(os.listdir(DATA_DIR)):
        if fname.startswith("batch_") and fname.endswith(".csv"):
            key = fname.replace(".csv", "")
            df  = pd.read_csv(os.path.join(DATA_DIR, fname), index_col=0)
            # columns of df are sample IDs; .T gives (n_samples, n_genes)
            batches[key]       = df.T.values
            batch_sample_ids   = df.columns.tolist()
            # FIX: record the position of each sample in the global label list
            batch_indices[key] = [all_sample_ids.index(sid) for sid in batch_sample_ids]
            print(f"Loaded {key}: {batches[key].shape}")

    true_labels_raw = labels_df.iloc[:, 0].values
    le          = LabelEncoder()
    true_labels = le.fit_transform(true_labels_raw)

    return batches, batch_indices, true_labels, le.classes_


# ─────────────────────────────────────────────
# 2. CLUSTERING PER BATCH
# ─────────────────────────────────────────────
def run_kmeans(X, k=N_CLUSTERS):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    return km.fit_predict(X)


def run_spectral(X, k=N_CLUSTERS):
    sc = SpectralClustering(
        n_clusters=k,
        affinity="rbf",
        gamma=0.01,
        random_state=42,
        n_init=10
    )
    return sc.fit_predict(X)


# ─────────────────────────────────────────────
# 3. CO-ASSOCIATION MATRIX  (vectorised — was O(n^2) loop)
# ─────────────────────────────────────────────
def co_association_matrix(labels):
    """
    C[i,j] = 1 if sample i and sample j are in the same cluster, else 0.
    Vectorised: outer equality check instead of nested for-loops.
    """
    labels = np.asarray(labels)
    C = (labels[:, None] == labels[None, :]).astype(float)
    return C


# ─────────────────────────────────────────────
# 4. ADMM CONSENSUS  (fixed x-update)
# ─────────────────────────────────────────────
def admm_consensus(co_matrices, n_clusters=N_CLUSTERS, n_iter=N_ITER, rho=RHO):
    """
    Consensus via ADMM.

    The original code computed the x-update as:
        Z_new = mean(C_b + dual_b / rho)
    This is WRONG — the duals start at zero so Z_new == mean(C_b) == Z on
    iteration 1, giving a primal residual of 0 and fake convergence.

    Correct ADMM primal consensus:
        Z  = mean(C_b)                            # global consensus variable
        z_b = argmin_{z_b} ||z_b - C_b||^2 + (rho/2)||z_b - Z + u_b||^2
            = (C_b + rho*(Z - u_b)) / (1 + rho)  # local z for each batch
        u_b += z_b - Z                            # scaled dual

    The primal residual is then ||Z - mean(z_b)||_F which is non-trivial.
    """
    n = co_matrices[0].shape[0]
    B = len(co_matrices)

    # Initialise global consensus and local copies + scaled duals
    Z     = np.mean(co_matrices, axis=0)
    z     = [C.copy() for C in co_matrices]   # local z_b
    u     = [np.zeros((n, n)) for _ in range(B)]  # scaled duals

    residuals = []

    for it in range(n_iter):
        # ── z_b update (closed-form) ──────────────────────────────────────
        for b in range(B):
            z[b] = (co_matrices[b] + rho * (Z - u[b])) / (1.0 + rho)
            z[b] = np.clip(z[b], 0.0, 1.0)

        # ── Z update: average of local z_b ───────────────────────────────
        Z_new = np.mean(z, axis=0)
        Z_new = np.clip(Z_new, 0.0, 1.0)
        Z_new = (Z_new + Z_new.T) / 2.0   # enforce symmetry

        # ── dual update ──────────────────────────────────────────────────
        for b in range(B):
            u[b] += z[b] - Z_new

        # ── primal residual: how much Z changed ──────────────────────────
        primal_res = np.linalg.norm(Z_new - Z, "fro")
        residuals.append(primal_res)
        Z = Z_new

        if (it + 1) % 10 == 0:
            print(f"  ADMM iter {it+1:3d}/{n_iter}  |  primal residual = {primal_res:.6f}")

        if primal_res < 1e-5:
            print(f"  Converged at iteration {it+1}")
            break

    return Z, residuals


def extract_clusters_from_consensus(Z, n_clusters=N_CLUSTERS):
    """Convert consensus matrix to labels via spectral clustering."""
    sc = SpectralClustering(
        n_clusters=n_clusters,
        affinity="precomputed",
        random_state=42,
        n_init=10
    )
    return sc.fit_predict(Z)


# ─────────────────────────────────────────────
# 5. EVALUATION  (fixed silhouette for consensus)
# ─────────────────────────────────────────────
def evaluate(name, pred_labels, true_labels, X_all, Z_consensus=None):
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)

    # FIX: for ADMM consensus use 1-Z as a distance matrix for silhouette;
    # for baseline use the raw feature matrix
    try:
        if Z_consensus is not None:
            dist_matrix = 1.0 - Z_consensus
            np.fill_diagonal(dist_matrix, 0.0)
            sil = silhouette_score(dist_matrix, pred_labels, metric="precomputed")
        else:
            sil = silhouette_score(X_all, pred_labels)
    except Exception:
        sil = float("nan")

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  ARI  (↑ better, max 1): {ari:.4f}")
    print(f"  NMI  (↑ better, max 1): {nmi:.4f}")
    print(f"  Silhouette (↑ better) : {sil:.4f}")
    return {"name": name, "ARI": ari, "NMI": nmi, "Silhouette": sil}


# ─────────────────────────────────────────────
# 6. PLOTS
# ─────────────────────────────────────────────
def plot_pca(X_all, labels_dict):
    pca  = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_all)
    var  = pca.explained_variance_ratio_

    n_plots = len(labels_dict)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]

    for ax, (name, labels) in zip(axes, labels_dict.items()):
        sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                        c=labels, cmap="tab10", s=30, alpha=0.8)
        ax.set_title(name, fontsize=11)
        ax.set_xlabel(f"PC1 ({var[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({var[1]*100:.1f}%)")
        plt.colorbar(sc, ax=ax, label="Cluster")

    plt.suptitle("PCA Projection", fontsize=13, y=1.02)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "pca_clusters.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"\nPCA plot saved → {path}")
    plt.close()


def plot_consensus_matrix(Z, title="Consensus Co-association Matrix"):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(Z, cmap="Blues", ax=ax, xticklabels=False, yticklabels=False,
                vmin=0, vmax=1, cbar_kws={"label": "Co-association score"})
    ax.set_title(title, fontsize=12)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "consensus_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Consensus matrix plot saved → {path}")
    plt.close()


def plot_admm_residuals(residuals):
    plt.figure(figsize=(6, 3))
    plt.plot(residuals, color="steelblue", linewidth=1.5)
    plt.xlabel("Iteration")
    plt.ylabel("Primal Residual (Frobenius norm)")
    plt.title("ADMM Convergence")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "admm_convergence.png")
    plt.savefig(path, dpi=150)
    print(f"ADMM convergence plot saved → {path}")
    plt.close()


def plot_metrics(results):
    df = pd.DataFrame(results).set_index("name")
    df[["ARI", "NMI", "Silhouette"]].plot(
        kind="bar", figsize=(8, 4), colormap="Set2", edgecolor="white"
    )
    plt.title("Clustering Method Comparison")
    plt.ylabel("Score")
    plt.xticks(rotation=25, ha="right")
    plt.ylim(-0.1, 1.1)
    plt.axhline(0, color="gray", linewidth=0.7)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, "metrics_comparison.png")
    plt.savefig(path, dpi=150)
    print(f"Metrics comparison plot saved → {path}")
    plt.close()


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # 1. Load data
    batches, batch_indices, true_labels, class_names = load_batches()
    batch_names = list(batches.keys())

    # FIX: stack in the same order as load_batches and reorder true_labels accordingly
    X_all          = np.vstack([batches[b] for b in batch_names])
    global_indices = []
    for b in batch_names:
        global_indices.extend(batch_indices[b])
    true_labels_aligned = true_labels[global_indices]

    print(f"\nTotal samples: {X_all.shape[0]}, Features: {X_all.shape[1]}")
    print(f"Classes: {class_names}")

    # 2. Per-batch K-means + co-association matrices
    print("\n--- Per-batch K-means clustering ---")
    km_labels_list = []
    km_co_matrices = []
    for bname in batch_names:
        lbl = run_kmeans(batches[bname])
        km_labels_list.append(lbl)
        km_co_matrices.append(co_association_matrix(lbl))
        print(f"  {bname}: clusters found = {np.unique(lbl)}")

    print("\n--- Per-batch Spectral clustering ---")
    sc_labels_list = []
    sc_co_matrices = []
    for bname in batch_names:
        lbl = run_spectral(batches[bname])
        sc_labels_list.append(lbl)
        sc_co_matrices.append(co_association_matrix(lbl))
        print(f"  {bname}: clusters found = {np.unique(lbl)}")

    # 3. Embed local co-association matrices into global n_total x n_total space
    n_total     = X_all.shape[0]
    batch_sizes = [batches[b].shape[0] for b in batch_names]
    offsets     = np.cumsum([0] + batch_sizes)

    def pad_coassoc(C, b_idx):
        C_global = np.zeros((n_total, n_total))
        s, e = offsets[b_idx], offsets[b_idx + 1]
        C_global[s:e, s:e] = C
        return C_global

    all_co_global = []
    for b_idx in range(len(batch_names)):
        all_co_global.append(pad_coassoc(km_co_matrices[b_idx], b_idx))
        all_co_global.append(pad_coassoc(sc_co_matrices[b_idx], b_idx))

    # 4. ADMM consensus
    print(f"\n--- ADMM Consensus ({N_ITER} iterations, rho={RHO}) ---")
    Z_consensus, residuals = admm_consensus(
        all_co_global, n_clusters=N_CLUSTERS, n_iter=N_ITER, rho=RHO
    )

    # 5. Extract labels from consensus matrix
    consensus_labels = extract_clusters_from_consensus(Z_consensus, N_CLUSTERS)

    # 6. Baseline: K-means on all data
    print("\n--- Baseline: K-means on full data ---")
    baseline_labels = run_kmeans(X_all)

    # 7. Evaluate
    results = []
    results.append(evaluate("Baseline K-means (all data)",
                             baseline_labels, true_labels_aligned, X_all))
    results.append(evaluate("ADMM Consensus Clustering",
                             consensus_labels, true_labels_aligned, X_all,
                             Z_consensus=Z_consensus))

    # 8. Plots
    plot_consensus_matrix(Z_consensus, "ADMM Consensus Co-association Matrix")
    plot_admm_residuals(residuals)
    plot_pca(X_all, {
        "True Labels":      true_labels_aligned,
        "Baseline K-means": baseline_labels,
        "ADMM Consensus":   consensus_labels,
    })
    plot_metrics(results)

    print("\nDone! All plots saved to the 'plots/' folder.")


if __name__ == "__main__":
    main()


Loaded batch_1: (52, 300)
Loaded batch_2: (52, 300)
Loaded batch_3: (51, 300)

Total samples: 155, Features: 300
Classes: ['Basal' 'Her2' 'Luminal A' 'Luminal B' 'None (normal)' 'unknown']

--- Per-batch K-means clustering ---


C:\Users\yasha\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\yasha\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\yasha\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\yasha\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\yasha\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

  batch_1: clusters found = [0 1 2 3 4 5]
  batch_2: clusters found = [0 1 2 3 4 5]


D:\c programming\proj\.venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\c programming\proj\.venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


  batch_3: clusters found = [0 1 2 3 4 5]

--- Per-batch Spectral clustering ---


...

  batch_1: clusters found = [0 1 2 3 4 5]
  batch_2: clusters found = [0 1 2 3 4 5]


D:\c programming\proj\.venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\c programming\proj\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:301: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
D:\c programming\proj\.venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


  batch_3: clusters found = [0 1 2 3 4 5]

--- ADMM Consensus (50 iterations, rho=1.0) ---
  Converged at iteration 1

--- Baseline: K-means on full data ---

  Baseline K-means (all data)
  ARI  (↑ better, max 1): 0.6034
  NMI  (↑ better, max 1): 0.6929
  Silhouette (↑ better) : 0.1699

  ADMM Consensus Clustering
  ARI  (↑ better, max 1): 0.0918
  NMI  (↑ better, max 1): 0.3299
  Silhouette (↑ better) : 0.0962


D:\c programming\proj\.venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Consensus matrix plot saved → data\plots\consensus_matrix.png
ADMM convergence plot saved → data\plots\admm_convergence.png

PCA plot saved → data\plots\pca_clusters.png
Metrics comparison plot saved → data\plots\metrics_comparison.png

Done! All plots saved to the 'plots/' folder.
